# Image EDA (OpenCV)

OpenCV scan of `data/FashionDataset` **train** and **test** images.

**Target:** `articleType`

This notebook:
1. Inventories image files vs CSV ids
2. Reads each JPEG with OpenCV (`width`, `height`)
3. Plots overall image-size distributions
4. Breaks **counts** and **image size** down by `articleType` (train labels; test CSV has no `articleType` values)


## Setup


In [1]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if ROOT.name == "eda":
    ROOT = ROOT.parent

TRAIN_IMG_DIR = ROOT / "data" / "FashionDataset" / "train" / "images_train"
TEST_IMG_DIR = ROOT / "data" / "FashionDataset" / "test" / "images_test"
TRAIN_CSV = ROOT / "data" / "FashionDataset" / "train" / "styles_train.csv"
TEST_CSV = ROOT / "data" / "FashionDataset" / "test" / "styles_prediction.csv"

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 80)
px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = px.colors.qualitative.Set2

print("OpenCV", cv2.__version__)
print("train images:", TRAIN_IMG_DIR.exists(), TRAIN_IMG_DIR)
print("test images:", TEST_IMG_DIR.exists(), TEST_IMG_DIR)


OpenCV 5.0.0
train images: True /Users/nhan.ngo/rmit/COSC2753-Project/data/FashionDataset/train/images_train
test images: True /Users/nhan.ngo/rmit/COSC2753-Project/data/FashionDataset/test/images_test


/Users/nhan.ngo/rmit/COSC2753-Project/eda/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## File inventory


In [2]:
def list_jpegs(folder: Path) -> pd.DataFrame:
    rows = []
    for p in folder.glob("*.jpg"):
        rows.append({"id": p.stem, "path": str(p), "bytes": p.stat().st_size})
    return pd.DataFrame(rows)


train_files = list_jpegs(TRAIN_IMG_DIR).assign(split="train")
test_files = list_jpegs(TEST_IMG_DIR).assign(split="test")
files = pd.concat([train_files, test_files], ignore_index=True)

train_meta = pd.read_csv(TRAIN_CSV, usecols=["id"])
test_meta = pd.read_csv(TEST_CSV, usecols=["id"])
train_meta["id"] = train_meta["id"].astype(str)
test_meta["id"] = test_meta["id"].astype(str)

inventory = pd.DataFrame(
    {
        "split": ["train", "test"],
        "n_csv_ids": [len(train_meta), len(test_meta)],
        "n_images": [len(train_files), len(test_files)],
        "csv_without_image": [
            len(set(train_meta["id"]) - set(train_files["id"])),
            len(set(test_meta["id"]) - set(test_files["id"])),
        ],
        "image_without_csv": [
            len(set(train_files["id"]) - set(train_meta["id"])),
            len(set(test_files["id"]) - set(test_meta["id"])),
        ],
        "id_overlap_train_test": [
            len(set(train_files["id"]) & set(test_files["id"])),
            len(set(train_files["id"]) & set(test_files["id"])),
        ],
    }
)
display(inventory)

missing_train_imgs = sorted(set(train_meta["id"]) - set(train_files["id"]))
print("Train CSV ids with no image:", missing_train_imgs)
print("File size (KB) by split")
display(files.assign(kb=files["bytes"] / 1024).groupby("split")["kb"].describe().round(2))


,split,n_csv_ids,n_images,csv_without_image,image_without_csv,id_overlap_train_test
0,train,38617,38612,5,0,0
1,test,5829,5829,0,0,0


Train CSV ids with no image: ['12347', '39401', '39403', '39410', '39425']
File size (KB) by split


,count,mean,std,min,25%,50%,75%,max
split,,,,,,,,
test,5829.0,14.39,8.57,0.57,4.42,16.19,19.77,111.26
train,38612.0,12.19,9.82,0.47,2.12,13.51,19.89,107.77


## OpenCV scan

For every image, OpenCV records:

- `height`, `width`, `channels` — geometry
- `brightness` — mean grayscale intensity
- `contrast` — grayscale standard deviation
- `sharpness` — variance of the Laplacian (higher = sharper)
- `saturation` — mean HSV S channel
- `b_mean`, `g_mean`, `r_mean` — mean BGR colour
- `white_bg_ratio` — share of near-white pixels (studio background)
- `edge_density` — share of Canny edge pixels


In [3]:
def analyze_image(path: str) -> dict:
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        return {
            "path": path,
            "ok": False,
            "height": np.nan,
            "width": np.nan,
            "channels": np.nan,
            "brightness": np.nan,
            "contrast": np.nan,
            "sharpness": np.nan,
            "saturation": np.nan,
            "b_mean": np.nan,
            "g_mean": np.nan,
            "r_mean": np.nan,
            "white_bg_ratio": np.nan,
            "edge_density": np.nan,
        }

    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    b_mean, g_mean, r_mean, _ = cv2.mean(img)
    edges = cv2.Canny(gray, 80, 160)

    return {
        "path": path,
        "ok": True,
        "height": int(h),
        "width": int(w),
        "channels": int(img.shape[2]),
        "brightness": float(gray.mean()),
        "contrast": float(gray.std()),
        "sharpness": float(cv2.Laplacian(gray, cv2.CV_64F).var()),
        "saturation": float(hsv[:, :, 1].mean()),
        "b_mean": float(b_mean),
        "g_mean": float(g_mean),
        "r_mean": float(r_mean),
        "white_bg_ratio": float((np.min(img, axis=2) > 245).mean()),
        "edge_density": float((edges > 0).mean()),
    }


def scan_split(file_df: pd.DataFrame) -> pd.DataFrame:
    records = [analyze_image(p) for p in tqdm(file_df["path"], desc=file_df["split"].iloc[0])]
    stats = pd.DataFrame(records)
    return file_df.merge(stats, on="path", how="left")


train_stats = scan_split(train_files)
test_stats = scan_split(test_files)
img_stats = pd.concat([train_stats, test_stats], ignore_index=True)
print(img_stats.groupby("split")["ok"].agg(["sum", "count"]))
img_stats.head()


test: 100%|██████████| 5829/5829 [00:02<00:00, 2760.92it/s]

         sum  count
split              
test    5829   5829
train  38612  38612


,id,path,bytes,split,ok,height,width,channels,brightness,contrast,sharpness,saturation,b_mean,g_mean,r_mean,white_bg_ratio,edge_density
0,9733,/Users/nhan.ngo/rmit/COSC2753-Project/data/Fas...,1694,train,True,80,60,3,207.922917,59.384421,5387.465976,35.141250,199.710000,209.938333,207.158542,0.473125,0.227708
1,14147,/Users/nhan.ngo/rmit/COSC2753-Project/data/Fas...,10078,train,True,80,60,3,250.205208,16.943637,381.548333,0.514583,249.924583,250.233125,250.243542,0.899167,0.037917
2,6400,/Users/nhan.ngo/rmit/COSC2753-Project/data/Fas...,20042,train,True,80,60,3,211.031667,79.160240,2142.002499,0.313125,210.944583,211.028333,211.035000,0.745000,0.090417
3,34297,/Users/nhan.ngo/rmit/COSC2753-Project/data/Fas...,15617,train,True,80,60,3,220.799375,59.742178,1873.628783,23.395625,212.203125,220.559167,224.541250,0.652708,0.095625
4,24084,/Users/nhan.ngo/rmit/COSC2753-Project/data/Fas...,20203,train,True,80,60,3,238.915417,44.719971,1760.257708,8.360000,236.489375,238.338125,240.965208,0.866458,0.059375


## Image size distribution

Pixel geometry (`width` × `height`), aspect ratio, pixel area, and JPEG file size for train vs test. Unreadable files are listed first.


In [4]:
failed = img_stats.loc[~img_stats["ok"], ["split", "id", "path"]]
print(f"Unreadable images: {len(failed)}")
display(failed if len(failed) else "None")

ok_stats = img_stats.loc[img_stats["ok"]].copy()
ok_stats["area"] = ok_stats["width"] * ok_stats["height"]
ok_stats["aspect_ratio"] = ok_stats["width"] / ok_stats["height"]
ok_stats["resolution"] = (
    ok_stats["width"].astype(int).astype(str)
    + "×"
    + ok_stats["height"].astype(int).astype(str)
)
ok_stats["file_kb"] = ok_stats["bytes"] / 1024

size_counts = (
    ok_stats.groupby(["split", "resolution", "width", "height", "channels"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)
print("Unique pixel sizes")
display(size_counts)

print("Pixel / file size summary")
display(
    ok_stats.groupby("split")[["width", "height", "area", "aspect_ratio", "file_kb"]]
    .describe()
    .round(3)
)

res_order = (
    size_counts.groupby("resolution")["n"].sum().sort_values(ascending=False).index.tolist()
)
fig = px.bar(
    size_counts,
    x="resolution",
    y="n",
    color="split",
    barmode="group",
    category_orders={"resolution": res_order},
    title="Image pixel size (width × height) — train vs test",
    labels={"resolution": "Width × height (px)", "n": "Number of images", "split": "Split"},
)
fig.update_yaxes(type="log")
fig.show()

fig = px.scatter(
    size_counts,
    x="width",
    y="height",
    size="n",
    color="split",
    size_max=48,
    opacity=0.75,
    title="Width vs height (bubble = image count)",
    labels={"width": "Width (px)", "height": "Height (px)", "split": "Split", "n": "Count"},
)
fig.update_traces(marker=dict(sizemin=8))
fig.show()

for col, label in [
    ("width", "Width (px)"),
    ("height", "Height (px)"),
    ("area", "Pixel area (width × height)"),
    ("aspect_ratio", "Aspect ratio (width / height)"),
    ("file_kb", "JPEG file size (KB)"),
]:
    fig = px.histogram(
        ok_stats,
        x=col,
        color="split",
        barmode="overlay",
        opacity=0.65,
        nbins=40,
        title=f"{label} distribution — train vs test",
        labels={col: label, "split": "Split"},
    )
    fig.show()


Unreadable images: 0


'None'

Unique pixel sizes


,split,resolution,width,height,channels,n
9,train,60×80,60,80,3,38595
3,test,60×80,60,80,3,5823
4,train,53×80,53,80,3,8
5,train,60×60,60,60,3,5
0,test,53×80,53,80,3,2
1,test,54×80,54,80,3,2
2,test,60×79,60,79,3,2
8,train,60×77,60,77,3,2
6,train,60×75,60,75,3,1
7,train,60×76,60,76,3,1


Pixel / file size summary


width                                                height          \
         count    mean    std   min   25%   50%   75%   max    count    mean   
split                                                                          
test    5829.0  59.996  0.171  53.0  60.0  60.0  60.0  60.0   5829.0  80.000   
train  38612.0  59.999  0.101  53.0  60.0  60.0  60.0  60.0  38612.0  79.997   

                                               area                            \
         std   min   25%   50%   75%   max    count      mean     std     min   
split                                                                           
test   0.019  79.0  80.0  80.0  80.0  80.0   5829.0  4799.623  13.703  4240.0   
train  0.231  60.0  80.0  80.0  80.0  80.0  38612.0  4799.705  16.027  3600.0   

                                      aspect_ratio                            \
          25%     50%     75%     max        count  mean    std    min   25%   
split                                                                          
test   4800.0  4800.0  4800.0  4800.0       5829.0  0.75  0.002  0.662  0.75   
train  4800.0  4800.0  4800.0  4800.0      38612.0  0.75  0.003  0.662  0.75   

                          file_kb                                       \
        50%   75%    max    count    mean    std    min    25%     50%   
split                                                                    
test   0.75  0.75  0.759   5829.0  14.391  8.572  0.567  4.419  16.189   
train  0.75  0.75  1.000  38612.0  12.188  9.824  0.470  2.119  13.507   

                        
          75%      max  
split                   
test   19.774  111.262  
train  19.887  107.767

## Target: `articleType`

1. **Class counts** — number of train images in each `articleType` (test CSV has no labels, so this is train only).
2. **Image size** — `width` × `height` in pixels, compared between **train** and **test**.


In [5]:
TARGET = "articleType"

styles = pd.read_csv(TRAIN_CSV)
unnamed = [c for c in styles.columns if str(c).startswith("Unnamed")]
styles = styles.drop(columns=unnamed, errors="ignore")
styles["id"] = styles["id"].astype(str)

train_by_type = (
    ok_stats.loc[ok_stats["split"] == "train"]
    .merge(styles[["id", TARGET]], on="id", how="left")
)
train_by_type[TARGET] = train_by_type[TARGET].fillna("Missing")

class_counts = (
    train_by_type.groupby(TARGET, dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)
class_counts["pct"] = (100 * class_counts["n"] / class_counts["n"].sum()).round(2)

print(f"Train classes: {len(class_counts)}  |  train images: {int(class_counts['n'].sum())}")
display(class_counts)

order = class_counts[TARGET].tolist()
fig = px.bar(
    class_counts,
    x="n",
    y=TARGET,
    orientation="h",
    text="n",
    title="Number of train images per articleType",
    labels={"n": "Number of images", TARGET: "articleType"},
    height=max(520, 16 * len(class_counts) + 80),
    category_orders={TARGET: order[::-1]},
)
fig.update_traces(textposition="outside")
fig.show()


Train classes: 124  |  train images: 38612


,articleType,n,pct
115,Tshirts,6780,17.56
88,Shirts,3087,7.99
16,Casual Shoes,2683,6.95
121,Watches,2252,5.83
96,Sports Shoes,1991,5.16
...,...,...,...
44,Ipad,1,0.00
106,Ties and Cufflinks,1,0.00
7,Body Wash and Scrub,1,0.00
64,Lounge Tshirts,1,0.00


In [6]:
print("Train vs test image counts")
split_n = (
    ok_stats.groupby("split")
    .size()
    .rename("n_images")
    .reset_index()
)
display(split_n)

fig = px.bar(
    split_n,
    x="split",
    y="n_images",
    color="split",
    text="n_images",
    title="Number of images — train vs test",
    labels={"split": "Split", "n_images": "Number of images"},
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

size_compare = (
    ok_stats.groupby(["split", "resolution", "width", "height"])
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)
size_compare["pct"] = (
    100 * size_compare["n"] / size_compare.groupby("split")["n"].transform("sum")
).round(2)
print("width × height counts by split")
display(size_compare)

res_order = (
    size_compare.groupby("resolution")["n"].sum().sort_values(ascending=False).index.tolist()
)
fig = px.bar(
    size_compare,
    x="resolution",
    y="n",
    color="split",
    barmode="group",
    text="n",
    category_orders={"resolution": res_order},
    title="Image size (width × height) — train vs test",
    labels={"resolution": "Width × height (px)", "n": "Number of images", "split": "Split"},
)
fig.update_traces(textposition="outside")
fig.show()

fig = px.bar(
    size_compare,
    x="resolution",
    y="pct",
    color="split",
    barmode="group",
    text="pct",
    category_orders={"resolution": res_order},
    title="Image size share (width × height) — train vs test",
    labels={"resolution": "Width × height (px)", "pct": "% of split", "split": "Split"},
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.show()

fig = px.histogram(
    ok_stats,
    x="width",
    color="split",
    barmode="overlay",
    opacity=0.65,
    nbins=20,
    title="Width (px) — train vs test",
    labels={"width": "Width (px)", "split": "Split"},
)
fig.show()

fig = px.histogram(
    ok_stats,
    x="height",
    color="split",
    barmode="overlay",
    opacity=0.65,
    nbins=20,
    title="Height (px) — train vs test",
    labels={"height": "Height (px)", "split": "Split"},
)
fig.show()

fig = px.scatter(
    size_compare,
    x="width",
    y="height",
    size="n",
    color="split",
    size_max=48,
    opacity=0.8,
    title="Width vs height — train vs test (bubble = count)",
    labels={"width": "Width (px)", "height": "Height (px)", "split": "Split", "n": "Count"},
)
fig.update_traces(marker=dict(sizemin=8))
fig.show()


Train vs test image counts


,split,n_images
0,test,5829
1,train,38612


width × height counts by split


,split,resolution,width,height,n,pct
9,train,60×80,60,80,38595,99.96
3,test,60×80,60,80,5823,99.90
4,train,53×80,53,80,8,0.02
5,train,60×60,60,60,5,0.01
0,test,53×80,53,80,2,0.03
1,test,54×80,54,80,2,0.03
2,test,60×79,60,79,2,0.03
8,train,60×77,60,77,2,0.01
6,train,60×75,60,75,1,0.00
7,train,60×76,60,76,1,0.00
